# loading and plotting data

Import toolboxes required for MNE coding

In [ ]:
import numpy as np
import mne 
import ipywidgets as widgets
print('MNE activated successfully')

Load Sample Data files and read them, stored into 'raw'

In [ ]:
sample_data_folder = mne.datasets.sample.data_path()                                  # get the path to the sample dataset
sample_data_file = (
    sample_data_folder / "MEG" / "sample" / "sample_audvis_filt-0-40_raw.fif"         # get the path to the raw data file
)
raw = mne.io.read_raw_fif(sample_data_file)                                           # read the raw data file

print raw data and info on the data

In [ ]:
print(raw)                                                 # print the raw data object
print(raw.info)                                            # print the information about the raw data

use compute_psd command to show the PSD for each sensor type, with a max frequency of 50 (idk what the rest means yet)

In [ ]:
raw.compute_psd(fmax=50).plot(picks="data", exclude="bads", amplitude=False)

plot raw data with raw.plot

In [ ]:
raw.plot(duration=5, n_channels=30)

# artifact removal

using ica function used mainly for artifact removal from EEG/MEG data. this remove things like eye blinks,
eye movements and heartbeat artifacts, and muscle noise that contaminate the neural signal
the code doesn't explain this, but to exclude the selected component we use the 'exclude' command

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800)
ica.fit(raw)
ica.exclude = [1, 2]  # details on how we picked these are omitted here
ica.plot_properties(raw, picks=ica.exclude)

to actually remove selected components from the signal, we need to apply the function 'apply', which will zero out all excluded
components. it will reconstruct the M/EEG signals. it also requires an 'unmixing matrix' idk yet, and inverse-transform the data. 

the data needs to be loaded into memory before we can use the apply function. made a copy of the raw data, and plotted the 'excluded' version with the unprocessed version for comparison to show the artifact removal

In [ ]:
orig_raw = raw.copy()
raw.load_data()
ica.apply(raw)

# show some frontal channels to clearly illustrate the artifact removal
chs = [
    "MEG 0111",
    "MEG 0121",
    "MEG 0131",
    "MEG 0211",
    "MEG 0221",
    "MEG 0231",
    "MEG 0311",
    "MEG 0321",
    "MEG 0331",
    "MEG 1511",
    "MEG 1521",
    "MEG 1531",
    "EEG 001",
    "EEG 002",
    "EEG 003",
    "EEG 004",
    "EEG 005",
    "EEG 006",
    "EEG 007",
    "EEG 008",
]
chan_idxs = [raw.ch_names.index(ch) for ch in chs]
orig_raw.plot(order=chan_idxs, start=12, duration=4)
raw.plot(order=chan_idxs, start=12, duration=4)

# event detecting

The sample dataset includes several “STIM” channels that recorded electrical signals sent from the stimulus delivery computer (as brief DC shifts / squarewave pulses). These pulses (often called “triggers”) are used in this dataset to mark experimental events: stimulus onset, stimulus type, and participant response (button press). 

The individual STIM channels are combined onto a single channel, in such a way that voltage levels on that channel can be unambiguously decoded as a particular event type. On older Neuromag systems (such as that used to record the sample data) this summation channel was called STI 014, so we can pass that channel name to the mne.find_events function to recover the timing and identity of the stimulus events.



the first column is the 'sample number' (basically 1 recording of data from the samples per second of an MEG/EEG) and the last column is the 'integer event ID'. the integer event ID is used in a conjunction with an 'event dictionary' to describe what kind of condition was elicited during a specific event ID

Event dictionaries like this one are used when extracting epochs from continuous data; the / character in the dictionary keys allows pooling across conditions by requesting partial condition descriptors (i.e., requesting 'auditory' will select all epochs with Event IDs 1 and 2; requesting 'left' will select all epochs with Event IDs 1 and 3). An example of this is shown in the next section. There is also a convenient plot_events function for visualizing the distribution of events across the duration of the recording (to make sure event detection worked as expected). Here we will also make use of the Info attribute to get the sampling frequency of the recording (so our x-axis will be in seconds instead of in samples).

In [ ]:
events = mne.find_events(raw, stim_channel="STI 014")
print(events[:5])  # show the first 5

event_dict = {
    "auditory/left": 1,
    "auditory/right": 2,
    "visual/left": 3,
    "visual/right": 4,
    "smiley": 5,
    "buttonpress": 32,
}

There is also a convenient plot_events function for visualizing the distribution of events across the duration of the recording (to make sure event detection worked as expected). Here we will also make use of the Info attribute to get the sampling frequency of the recording (so our x-axis will be in seconds instead of in samples).

In [ ]:
fig = mne.viz.plot_events(
    events, event_id=event_dict, sfreq=raw.info["sfreq"], first_samp=raw.first_samp
)

For paradigms that are not event-related (e.g., analysis of resting-state data), you can extract regularly spaced (possibly overlapping) spans of data by creating events using mne.make_fixed_length_events and then proceeding with epoching as described in the next section

# Epochs

the raw data file can be used to extract the Epochs, which is .a distinct period of time, a point that marks a new development, or a reference point in science and computing. in this case, an Epoch is a distinct time frame surrounding our 'events' elicited (look above) for this, we use 'mne.epochs'. 

we can also specify data quality constraints, such as peak-to-peak signal amplitude. these values for amplitude are decided by your particular data set

using the event_id parameter, we can plug in our event dictionary made before to have labels for the integer event IDs. the tmin and tmax values are the time relative to each event at which to start and end each epoch. 

In [ ]:
reject_criteria = dict(
    mag=4000e-15,  # 4000 fT
    grad=4000e-13,  # 4000 fT/cm
    eeg=150e-6,  # 150 µV
    eog=250e-6,
)  # 250 µV

epochs = mne.Epochs(
    raw,
    events,
    event_id=event_dict,
    tmin=-0.2,
    tmax=0.5,
    reject=reject_criteria,
    preload=True,
)

Next we’ll pool across left/right stimulus presentations so we can compare auditory versus visual responses. To avoid biasing our signals to the left or right, we’ll use equalize_event_counts first to randomly sample epochs from each condition to match the number of epochs present in the condition with the fewest good epochs.

In [ ]:
conds_we_care_about = ["auditory/left", "auditory/right", "visual/left", "visual/right"]
epochs.equalize_event_counts(conds_we_care_about)  # this operates in-place
aud_epochs = epochs["auditory"]
vis_epochs = epochs["visual"]
del raw, epochs  # free up memory

Like Raw objects, Epochs objects also have a number of built-in plotting methods. One is plot_image, which shows each epoch as one row of an image map, with color representing signal magnitude; the average evoked response and the sensor location are shown below the image:

In [ ]:
aud_epochs.plot_image(picks=["MEG 1332", "EEG 021"])

# Time-frequency analysis

The mne.time_frequency submodule provides implementations of several algorithms to compute time-frequency representations, power spectral density, and cross-spectral density. 

Time-frequency analysis of MEG and EEG data evaluates how brain wave power and frequency change over time using primary methods like Morlet wavelets, the Short-Time Fourier Transform (STFT), and the Hilbert transform. It allows researchers to capture dynamic neural oscillations that standard time-domain averaging hides


### Core Concepts

Evoked vs. Induced Responses: Evoked activity is phase-locked to a stimulus (visible in ERPs/ERFs), while induced activity reflects non-phase-locked power changes (like gamma band shifts) that require time-frequency mapping to detect.

Neural Frequency Bands: Delta (0.5–2 Hz), theta (4–7 Hz), alpha (8–12 Hz), beta (13–30 Hz), and gamma (>30 Hz) track distinct cognitive states, attention, and sensory processing.

### Common Decomposition Methods

Morlet Wavelets: Uses a sliding Gaussian-shaped window that adapts its length, offering high frequency resolution at low frequencies and high time resolution at high frequencies.

Multitaper / Short-Time Fourier Transform (STFT): Applies fixed or variable sliding time windows multiplied by tapers to control spectral leakage and manage smoothing.#

Hilbert Transform: Filters signals into distinct frequency bands and extracts the analytic signal envelope to measure instantaneous power.

In [ ]:
frequencies = np.arange(7, 30, 3)
power = aud_epochs.compute_tfr(
    "morlet", n_cycles=2, return_itc=False, freqs=frequencies, decim=3, average=True
)
power.plot(["MEG 1332"])

# estimating evoked responses

Now that we have our conditions in aud_epochs and vis_epochs, we can get an estimate of evoked responses to auditory versus visual stimuli by averaging together the epochs in each condition. This is as simple as calling the 'average' method on the Epochs object, and then using a function from the 'mne.viz' module to compare the global field power for each sensor type of the two Evoked objects:

In [ ]:
aud_evoked = aud_epochs.average()
vis_evoked = vis_epochs.average()

mne.viz.plot_compare_evokeds(
    dict(auditory=aud_evoked, visual=vis_evoked),
    legend="upper left",
    show_sensors="upper right",
)

We can also get a more detailed view of each Evoked object using other plotting methods such as 'plot_joint' or 'plot_topomap'. Here we’ll examine just the EEG channels, and see the classic auditory evoked N100-P200 pattern over dorso-frontal electrodes, then plot scalp topographies at some additional arbitrary times:

Evoked objects can also be combined to show contrasts between conditions, using the mne.combine_evoked function. A simple difference can be generated by passing weights=[1, -1]. We’ll then plot the difference wave at each sensor using plot_topo:


In [ ]:
aud_evoked.plot_joint(picks="eeg")
aud_evoked.plot_topomap(times=[0.0, 0.08, 0.1, 0.12, 0.2], ch_type="eeg")

evoked_diff = mne.combine_evoked([aud_evoked, vis_evoked], weights=[1, -1])
evoked_diff.pick(picks="mag").plot_topo(color="r", legend=False)


# inverse modelling

Finally, we can estimate the origins of the evoked activity by projecting the sensor data into this subject’s source space (a set of points either on the cortical surface or within the cortical volume of that subject, as estimated by structural MRI scans). MNE-Python supports lots of ways of doing this (dynamic statistical parametric mapping, dipole fitting, beamformers, etc.); here we’ll use minimum-norm estimation (MNE) to generate a continuous map of activation constrained to the cortical surface. MNE uses a linear inverse operator to project EEG+MEG sensor measurements into the source space. The inverse operator is computed from the forward solution for this subject and an estimate of the covariance of sensor measurements. For this tutorial we’ll skip those computational steps and load a pre-computed inverse operator from disk (it’s included with the sample data). Because this “inverse problem” is underdetermined (there is no unique solution), here we further constrain the solution by providing a regularization parameter specifying the relative smoothness of the current estimates in terms of a signal-to-noise ratio (where “noise” here is akin to baseline activity level across all of cortex).

In [ ]:
# load inverse operator
inverse_operator_file = (
    sample_data_folder / "MEG" / "sample" / "sample_audvis-meg-oct-6-meg-inv.fif"
)
inv_operator = mne.minimum_norm.read_inverse_operator(inverse_operator_file)
# set signal-to-noise ratio (SNR) to compute regularization parameter (λ²)
snr = 3.0
lambda2 = 1.0 / snr**2
# generate the source time course (STC)
stc = mne.minimum_norm.apply_inverse(
    vis_evoked, inv_operator, lambda2=lambda2, method="MNE"
)  # or dSPM, sLORETA, eLORETA


Finally, in order to plot the source estimate on the subject’s cortical surface we’ll also need the path to the sample subject’s structural MRI files (the subjects_dir):

In [ ]:
# path to subjects' MRI files
subjects_dir = sample_data_folder / "subjects"
# plot the STC
stc.plot(
    initial_time=0.1, hemi="split", views=["lat", "med"], subjects_dir=subjects_dir
)